# GPU runner (Kaggle or Colab)

Clones the repo, installs it, and runs one training command. Results end up in `results.zip`; unzip it into `reports/results/` locally and commit.

**Why a notebook at all?** The training code lives in the repository (`src/`, `scripts/`). This notebook is only a thin launcher, so the free GPUs on Kaggle/Colab run *exactly* the committed code, and anyone can reproduce the results with the same command.

**One-time setup**
- Only if the repo is private: a read-only GitHub token stored as a secret named `GITHUB_TOKEN` (Kaggle: Add-ons → Secrets; Colab: the key icon). Public repo: nothing to do.
- **Kaggle:** Settings → Accelerator **GPU T4 x2** (P100 is retired), Internet **on** (needs a phone-verified account), and *Add Input* → competition **asap-aes**.
- **Colab:** Runtime → change runtime type → **T4 GPU**. Put `training_set_rel3.tsv` (or its .zip) in Google Drive under `MyDrive/asap-aes/`.

For long runs on Kaggle, use **Save Version → Save & Run All** so the run continues with the tab closed. (Running cells in the editor is an interactive session, which stops when the tab closes.)

This notebook never prints essay text. The competition rules forbid sharing it.

## 1. Choose what to run
This is the only cell you normally edit. `CONFIG` picks the model (each YAML in `configs/` documents its settings), and `FOLDS` picks which of the 5 cross-validation folds to train.

In [ ]:
# ---- What to run ----
BRANCH = "main"                          # git branch to clone: lets you test changes on a branch before merging
CONFIG = "configs/deberta_v3_base.yaml"  # or "configs/bert_base.yaml" for the BERT comparison
FOLDS = "0 1 2 3 4"                      # all 5 folds = the full evaluation; fewer for a partial or resumed run
EXTRA = ""            # e.g. "--seed 43" or "--limit 64 --epochs 1" for a quick test

## 2. Detect the platform and get the code
Kaggle and Colab differ in where files live, how secrets are read, and where the data comes from, so the notebook first detects which one it is on. The repository is then cloned fresh, so every run uses the latest committed code and nothing left over from an earlier run.

In [ ]:
import glob, os, shutil, subprocess, zipfile

ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ  # not os.path.exists("/kaggle"): Colab has that folder too
REPO = "Dilip0552/essay-grader-lstm-to-llm"
# WORK = the folder the platform keeps after the run (Kaggle saves /kaggle/working as the notebook's output).
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
CODE = "/tmp/repo"  # outside WORK so the token-bearing clone is never saved as notebook output

# A GitHub token is only needed to clone a private repo. It is read from the platform's secret store,
# never typed into the notebook, so it cannot leak through the saved notebook or its output.
token = ""  # only needed while the repo is private
try:
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    else:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
except Exception:  # no secret set (public repo): clone without a token
    pass
if not ON_KAGGLE:
    # On Colab the dataset is read from Google Drive (Kaggle attaches it as an input instead).
    from google.colab import drive
    drive.mount("/content/drive")

os.chdir(WORK)  # leave CODE before deleting it, so re-running this cell works
shutil.rmtree(CODE, ignore_errors=True)  # fresh clone every time: no stale code from an earlier run
r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, f"https://{token + '@' if token else ''}github.com/{REPO}.git", CODE],
                   capture_output=True, text=True)
if r.returncode:
    # Show git's error, but with the token masked so it never appears in the output.
    raise RuntimeError(r.stderr.replace(token, "***") if token else r.stderr)
# Remove the token from the clone's saved remote URL as well.
subprocess.run(["git", "-C", CODE, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], check=True)
os.chdir(CODE)  # all later commands run from the repository root
# Print the exact commit being run, so the log shows which code produced the results.
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

## 3. Install the project
Installs the listed dependencies and the project itself in editable mode (`-e .`), so the scripts can `import aes`. Packages already on Kaggle/Colab (such as PyTorch) are kept, because the requirements only set minimum versions.

In [ ]:
!pip install -q -r requirements.txt -e .

## 4. Locate the ASAP data and build the folds
The dataset is never stored in the repository (competition rules), so the notebook finds it on the platform: Kaggle's attached competition input, or the Google Drive folder on Colab. `scripts/download_data.py` then fetches the published fold ids and checks that every fold can be built.

In [ ]:
# Find the ASAP data (Kaggle input or Google Drive), unzip if needed
roots = ["/kaggle/input"] if ON_KAGGLE else ["/content/drive/MyDrive/asap-aes"]
# Search recursively: Kaggle's input folder layout can change, and the file may be a .tsv or a .tsv.zip.
hits = [p for r in roots for p in glob.glob(f"{r}/**/training_set_rel3.tsv*", recursive=True)]
assert hits, f"training_set_rel3.tsv not found under {roots}"
tsv = next((p for p in hits if p.endswith(".tsv")), None)  # prefer the plain .tsv if both exist
if tsv is None:
    # Only the zip was found: extract it to a temporary folder (input folders are read-only).
    zipfile.ZipFile(hits[0]).extractall("/tmp/asap")
    tsv = "/tmp/asap/training_set_rel3.tsv"
# aes.data.tsv_path() reads this variable, so the training scripts find the data without code changes.
os.environ["ASAP_TSV"] = tsv
!python scripts/download_data.py

## 5. Train
Each fold is an independent training run with its own train/dev/test split, so folds can run at the same time on different GPUs. Kaggle's *T4 x2* gives two GPUs, which halves the total time; with one GPU the folds simply run one after another. Each GPU writes a log file, and at the end the cell prints a summary of every epoch and the test scores.

In [ ]:
# Runs folds in parallel when there are 2 GPUs (Kaggle T4 x2), sequentially otherwise
import torch
n_gpu = max(torch.cuda.device_count(), 1)
folds = FOLDS.split()
procs = []
for g in range(n_gpu):
    mine = folds[g::n_gpu]  # round-robin: GPU 0 gets folds 0, 2, 4 and GPU 1 gets folds 1, 3
    if mine:
        cmd = f"python scripts/train_transformer.py --config {CONFIG} --folds {' '.join(mine)} {EXTRA}"
        log = open(f"{WORK}/train_gpu{g}.log", "w")  # saved in WORK, so the logs are kept with the output
        # CUDA_VISIBLE_DEVICES makes each process see only its own GPU, so the two runs never share a card.
        procs.append(subprocess.Popen(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT,
                                      env={**os.environ, "CUDA_VISIBLE_DEVICES": str(g)}))
        print(f"GPU {g}: folds {mine}  (log: {WORK}/train_gpu{g}.log)")
codes = [p.wait() for p in procs]  # block until every fold has finished (nothing is printed until then)
for g in range(len(procs)):
    print(f"--- GPU {g} results ---")
    # Show only the useful lines: per-epoch summaries "{...}", per-fold QWK "[...]", and any error trace.
    print("".join(l for l in open(f"{WORK}/train_gpu{g}.log") if l.startswith(("{", "[", "Traceback", "  File", "RuntimeError", "ValueError"))))
assert not any(codes), f"exit codes {codes}: check the logs above"


## 6. Package the results
Zips `reports/results/`: the prediction files (essay ids and scores only), training histories and timing metadata. Model weights are never saved, so the zip stays small. On Kaggle, download `results.zip` from the notebook's **Output** tab; on Colab it downloads automatically. Unzip it into `reports/results/` locally and commit.

In [ ]:
shutil.make_archive(f"{WORK}/results", "zip", "reports/results")  # -> results.zip in the kept output folder
print("saved", f"{WORK}/results.zip")
if not ON_KAGGLE:
    # Colab deletes its files when the session ends, so trigger a browser download right away.
    from google.colab import files
    files.download(f"{WORK}/results.zip")